# Lab 02: Core ML & Retrieval Algorithms from Scratch (Pure NumPy)

**Companion Module:** [Module 13: Hands-On Algorithms from Scratch](../../curriculum/06-production-ai/13_algorithms_from_scratch.md)  
**Difficulty:** Intermediate | **Environment:** Pure Python 3.11+, NumPy

---

## 1. Problem Statement
Many data scientists rely exclusively on high-level library abstractions (`model.fit()`, `model.predict()`), leading to silent numerical bugs in production:
1. **Floating-point overflow:** Computing naive exponentials in Softmax leading to `NaN` crashes.
2. **Inefficient Top-K search:** Sorting entire $N$-element arrays ($O(N \log N)$) instead of linear selection ($O(N)$).
3. **Black-box gradient descent:** Inability to diagnose divergent or zero gradient updates.

---

## 2. Learning Objectives
By the end of this lab, you will be able to:
- Implement row-wise numerically stable Softmax and Cross-Entropy loss.
- Build vectorized batch gradient descent for linear regression from first principles.
- Implement an efficient $O(N)$ Top-K Cosine retrieval engine using `np.argpartition`.

---

## 3. Required Dependencies
```bash
pip install numpy
```


In [ ]:
import numpy as np
print(f"NumPy Version: {np.__version__}")


---

## 4. Dataset Description
Synthetic matrices designed to stress-test numerical boundaries:
- Unbounded logits with values ranging up to $+1000.0$ (triggering standard exponential overflow).
- 2D feature matrices $X \in \mathbb{R}^{4 \times 2}$ with known ground-truth affine targets $y = 1.0 x_1 + 2.0 x_2 + 1.0$.
- High-dimensional vector corpus representing document embeddings.


In [ ]:
# 5. Implementation Part 1: Numerically Stable Softmax & Cross-Entropy
def stable_softmax(logits: np.ndarray) -> np.ndarray:
    """Computes row-wise stable softmax by subtracting max logit before exponentiation."""
    assert logits.ndim == 2, f"Expected 2D array, got shape {logits.shape}"
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)

# Stress test with extreme values that would overflow naive exp()
extreme_logits = np.array([
    [1000.0, 1001.0, 1002.0],
    [-500.0, 0.0, 500.0]
])
probs = stable_softmax(extreme_logits)

assert np.allclose(probs.sum(axis=-1), 1.0)
assert not np.isnan(probs).any()
print("Stable Softmax Probabilities:\n", probs)


In [ ]:
# 5. Implementation Part 2: Vectorized Batch Gradient Descent (Linear Regression)
X = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [3.0, 4.0],
    [4.0, 5.0]
], dtype=float)

# Ground truth: y = 1*x1 + 2*x2 + 1.0
y = 1.0 * X[:, 0] + 2.0 * X[:, 1] + 1.0

n_samples, n_features = X.shape
weights = np.zeros(n_features, dtype=float)
bias = 0.0
lr = 0.02
epochs = 2000

for epoch in range(epochs):
    preds = X @ weights + bias
    diff = preds - y
    grad_w = (2.0 / n_samples) * (X.T @ diff)
    grad_b = (2.0 / n_samples) * float(np.sum(diff))
    weights -= lr * grad_w
    bias -= lr * grad_b

assert np.allclose(weights, [1.0, 2.0], atol=1e-2)
assert np.isclose(bias, 1.0, atol=1e-2)
print(f"Fitted Weights: {weights} | Fitted Bias: {bias:.4f}")


In [ ]:
# 5. Implementation Part 3: O(N) Top-K Cosine Retrieval via np.argpartition
def top_k_cosine(query: np.ndarray, corpus: np.ndarray, k: int = 2) -> tuple[np.ndarray, np.ndarray]:
    """Finds top-k cosine nearest neighbors in O(N) without full array sorting."""
    qn = max(float(np.linalg.norm(query)), 1e-12)
    cn = np.maximum(np.linalg.norm(corpus, axis=1, keepdims=True), 1e-12)

    sims = (corpus / cn) @ (query / qn)
    n = len(corpus)
    k = min(k, n)

    # Use argpartition to find top k elements in O(N)
    top_indices = np.argpartition(sims, -k)[-k:]
    # Sort only the k elements
    sorted_top = top_indices[np.argsort(-sims[top_indices])]
    return sorted_top, sims[sorted_top]

# Search corpus
doc_corpus = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.9, 0.1],
    [-1.0, 0.0]
])
query_vec = np.array([1.0, 0.0])

indices, scores = top_k_cosine(query_vec, doc_corpus, k=2)
assert list(indices) == [0, 2]
assert np.isclose(scores[0], 1.0)
print(f"Top-{len(indices)} Retrieved Doc Indices: {indices} with Cosine Scores: {scores}")


---

## 6. Results and Observations
1. **Softmax Stability:** The max-subtraction identity $\frac{e^{z_i - c}}{\sum e^{z_j - c}} = \frac{e^{z_i}}{\sum e^{z_j}}$ completely prevents floating-point overflow for arbitrarily large positive logits.
2. **Vectorized GD:** Matrix-vector products `$X^T @ diff$` eliminated sample-level loops, achieving fast convergence.
3. **Partition Retrieval:** `np.argpartition` achieves linear $O(N)$ selection time, making it scalable to million-vector retrieval systems.

---

## 7. Hands-on Exercises
- **Exercise 1:** Implement L2 regularization (Ridge penalty) inside the gradient descent loop.
- **Exercise 2:** Extend `stable_softmax` to calculate binary sigmoid when given 1D logit arrays.

---

## 8. Further Exploration
- See [Module 13: Algorithms from Scratch](../../curriculum/06-production-ai/13_algorithms_from_scratch.md) for complete implementations of K-Means and Reciprocal Rank Fusion.
